# Demo Data Generation


In [ ]:
from types import SimpleNamespace
import os
from datetime import datetime, timezone
import json
from IPython import get_ipython
import numpy as np
import plotly.graph_objects as go

from meta import update_meta

detect if runnig as notebook or script


In [2]:
if get_ipython():
    is_running_in_notebook = True
    print("executing as notebook ...")
else:
    is_running_in_notebook = False
    print("executing as script ...")

executing as notebook ...


## config


In [3]:
args = SimpleNamespace(
    data_path="data/demo/",
    input_size=2,
    output_size=3,
    num_samples=10_000,
    rel_noise=0.1,
    abs_noise=0.01,
)

## aux


### random polynomial


In [4]:
def rand_polynomial(*, dim: int, deg: int):
    r"""
    Returns a function representing a random polynomial of given input vector dimension and degree.

    $$ f(x) = \sum_{i=0}^{deg} \sum_{j=0}^{dim-1} c_{j,i} x_j^i $$

    """
    cs = np.random.randn(dim, deg + 1)

    def f(x):
        return sum(cs[j, i] * x[j] ** i for i in range(deg + 1) for j in range(dim))

    return f

### noise


In [5]:
def add_noise(y, rel_noise: float = 0.1, abs_noise: float = 0.0):
    rn = rel_noise * np.random.randn(*y.shape)
    an = abs_noise * np.random.randn(*y.shape)
    return y * (1 + rn) + an

### IO


In [6]:
def save(name, data):
    path = os.path.join(args.data_path, f"{name}.npz")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **data)

In [7]:
def load(name):
    path = os.path.join(args.data_path, f"{name}.npz")
    with np.load(path) as data:
        return {k: data[k] for k in data.keys()}

### show


In [8]:
def show_axes_projected_scatter(x, y, name=None):
    """Axes projected scatter plots across all combinations of input and output dimensions."""
    for i in range(len(x[0])):

        fig = go.Figure()
        for j in range(len(y[0])):
            fig.add_trace(
                go.Scatter(x=x[:, i], y=y[:, j], mode="markers", name=f"y[{j}]")
            )
        fig.update_layout(
            title=(f"{name}: " if name else "") + f"x[{i}] vs y",
            xaxis_title=f"x[{i}]",
            yaxis_title="y",
        )
        fig.show()


def show_axex_projected_box_plot(x, name=None):
    """Axes projected box plots across all input dimensions."""
    fig = go.Figure()

    for i in range(len(x[0])):
        fig.add_trace(go.Box(y=x[:, i], name=f"x[{i}]"))
    fig.update_layout(
        title=(f"{name}: " if name else "") + f"x distribution",
        yaxis_title=f"x",
    )
    fig.show()

In [9]:
def show(name):
    data = load(name)
    if "x" in data and "y" in data:
        show_axes_projected_scatter(data["x"], data["y"], name=name)
    if "x" in data and not "y" in data:
        show_axex_projected_box_plot(data["x"], name=name)

## generator


### fix function


In [10]:
fs = tuple(rand_polynomial(dim=args.input_size, deg=5) for _ in range(args.output_size))

In [11]:
def make_dataset(include_y=True, show=is_running_in_notebook):
    x = np.random.rand(args.num_samples, args.input_size).astype(np.float32) * 2 - 1

    if not include_y:
        return dict(x=x)
    else:
        y = np.zeros((args.num_samples, args.output_size), dtype=np.float32)
        for i, f in enumerate(fs):
            for n in range(args.num_samples):
                y[n, i] = f(x[n, :])
        y = add_noise(
            y,
            rel_noise=args.rel_noise,
            abs_noise=args.abs_noise,
        )
        return dict(x=x, y=y)

In [ ]:
def save_meta():
    user = os.getlogin()
    date = datetime.now(tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    meta = {
        "author": user,
        "date": date,
        "description": "Demo dataset based on a random polynomial function with noise.",
    } | {k: v for k, v in vars(args).items()}

    update_meta(args.data_path, meta)

## make & save


In [13]:
save_meta()
save("train", make_dataset())
save("test", make_dataset())
save("predict", make_dataset(include_y=False))

## verify


In [14]:
show("train")
show("test")
show("predict")